# Retrieval Improvement in Our HR RAG System

Interview-style notes covering **what we discussed**, **methods to improve embedding retrieval**, **from-scratch code**, **how we applied each method in this repo**, and **measured impact**.

**Corpus:** FAQ.docx · Terms & Conditions · Refund/Cancellation Policy  
**Production stack (best overall):** `qwen3-embedding:0.6b` + hybrid RRF (dense **0.3** / BM25 **0.7**) + cross-encoder rerank → top_k=5

## 0. Why retrieval mattered

Two user-facing failures drove this work:

1. **Subscription / multi-chunk answers** — free vs paid tiers live in different FAQ chunks; retrieval had to surface *both*.
2. **“How can I join Vedic Marriage?”** — retrieved *contact members* FAQ instead of *registration* (duplicate `Q1` labels + bad section metadata).

**Key distinction:**

| Layer | Job |
|-------|-----|
| **Retrieval** | Put the right chunks in top-k |
| **Generation** | Write the answer from those chunks |

We measure retrieval with **qrels** (query → gold chunks) and metrics: **Recall@k, MRR, nDCG@k, Precision@k**.  
Qrels are an **eval answer key**, not runtime routing — production never hardcodes “if join then FAQ-Q1”.

## 1. Methods map — what we tried

| # | Method | Tried? | Outcome |
|---|--------|--------|---------|
| 1 | Structure-aware chunking (FAQ / legal) | Yes | Fixed join bug; FAQ MRR ~0.95 |
| 2 | Chunk headers in text + IDs (`faq_id`, `legal_id`) | Yes | Precise eval + better embeddings |
| 3 | Embedding model comparison | Yes | Models similar after fair rebuild; use qwen3 |
| 4 | Wider candidate set (`fetch_k`) | Yes | Needed so gold isn’t missed before rerank |
| 5 | Cross-encoder reranking | Yes | Large quality jump vs embedding-only |
| 6 | Intent-aware scoring (registration / policy) | Yes | Join fixed; FAQ doesn’t drown legal |
| 7 | Hybrid BM25 + dense (weighted RRF) | Yes | Terms ↑; overall best with 30/70 |
| 8 | Stronger qrels / `gold_*` IDs | Yes | Fair measurement |
| 9 | Paraphrase + multi-chunk composite qrels | Yes | Harder, more realistic tests |
| 10 | Policy-only hybrid | Tried | Weaker — missed many Terms queries |
| — | Query rewrite / HyDE / fine-tuned embeds | Not yet | Future |

Each method below: **definition → interview code → our project → impact**.

## 2. Structure-aware chunking

### Definition
Split documents by **semantic boundaries** (FAQ Q&A, “1.1 Clause…”, titled sections) instead of fixed 1000-char windows. Better chunks ⇒ better embeddings ⇒ better retrieval.

### Interview-style code (from scratch)

In [ ]:
import re
from dataclasses import dataclass

@dataclass
class Chunk:
    text: str
    section: str
    chunk_id: str

def split_faq(raw: str) -> list[Chunk]:
    """One chunk per Qn. — interview answer for FAQ chunking."""
    section = "FAQ"
    chunks = []
    # Split before lines like Q1. / Q12.
    parts = re.split(r"\n(?=Q\d+\.)", raw.strip())
    for part in parts:
        part = part.strip()
        if not part:
            continue
        if re.match(r"^Q\d+\.", part):
            q = re.match(r"^(Q\d+)", part).group(1)
            slug = re.sub(r"[^a-z0-9]+", "-", section.lower()).strip("-")
            chunks.append(Chunk(
                text=f"{section}\n\n{part}",
                section=section,
                chunk_id=f"{slug}-{q}",
            ))
        else:
            # Non-Q block may be a section header (Registration, Login, …)
            lines = [l.strip() for l in part.splitlines() if l.strip()]
            if lines:
                section = lines[-1]
    return chunks

# Demo
demo = """Getting started:Registration\n\nQ1. How do I register?\nAns. Fill the form.\n\nLogin & Password\n\nQ1. How do I login?\nAns. Use email."""
for c in split_faq(demo):
    print(c.chunk_id, "|", c.text.split("\n")[0], "|", c.text.split("\n")[2][:40])

### Applied in our project
- `app/services/document_service.py` → `_split_faq_document`, `_split_legal_by_markers`, `_split_legal_by_title_headers`
- Line-by-line FAQ parse so **Login / Profile / Payment** sections don’t all inherit `Getting started:Registration`
- Legal clauses get headers prepended into `page_content`

### Impact
- Fixed **six different `Q1` chunks** colliding on one section
- “Join Vedic Marriage” retrieves `registration-Q1` (sign-up), not member-contact
- FAQ Recall/MRR ≈ **0.95 / 0.94** after fair rebuild

## 3. Chunk IDs + headers in text

### Definition
Stable IDs (`faq_id`, `legal_id`) and visible titles in the embedded text. Helps **retrieval** (more signal) and **evaluation** (exact match to gold).

### Interview-style code

In [ ]:
def ensure_header(body: str, *headers: str) -> str:
    """Guarantee titles appear at the start of the chunk text."""
    text = body.strip()
    head = text.lower()[:160]
    prefix = []
    for h in headers:
        h = (h or "").strip()
        if h and h.lower() not in head and h not in prefix:
            prefix.append(h)
    return ("\n".join(prefix) + "\n\n" + text) if prefix else text

raw_clause = "1.1. No Refunds: All payments are final..."
print(ensure_header(raw_clause, "1. Refund Policy", "1.1. No Refunds"))

### Applied / impact
- Metadata: `faq_id=registration-Q1`, `legal_id=5-force-majeure`, `legal_heading`, …
- Benchmark matches `gold_faq_id` / `gold_legal_id` instead of ambiguous `Q1\\.`
- Legal MRR measurement became trustworthy after strengthen script

## 4. Embedding model comparison

### Definition
Dense retrieval: embed query & chunks, rank by cosine similarity / inner product. Model choice changes geometry of “similar”.

### Interview-style code

In [ ]:
import numpy as np

def cosine(a: np.ndarray, b: np.ndarray) -> float:
    return float(a @ b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

def dense_topk(query_vec, chunk_vecs, texts, k=5):
    """Rank chunks by cosine similarity to the query vector."""
    scores = [cosine(query_vec, v) for v in chunk_vecs]
    order = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    return [(texts[i], scores[i]) for i in order[:k]]

# Toy 2D example
q = np.array([1.0, 0.2])
chunks = {
    "register FAQ": np.array([0.95, 0.1]),
    "contact members": np.array([0.4, 0.9]),
    "refund 1.1": np.array([0.1, 0.2]),
}
print(dense_topk(q, list(chunks.values()), list(chunks.keys()), k=3))

### Applied / impact
- Compared: `ollama_nomic`, `ollama_bge-m3`, `hf_bge-base`, `ollama_qwen3-0.6b`
- After **fair rebuild** (all stores with `faq_id`): scores close; FAQ strong (~95%), Terms/Refund weaker before hybrid
- **Default store:** `ollama_qwen3-0.6b`
- Lesson: don’t compare models if only one index has new metadata

## 5. Wider candidate set (`fetch_k`) then cut to top_k

### Definition
Embedding rank of the gold legal chunk is often **8–20**, not 1–5. Fetch more candidates, then rerank / fuse, finally keep top_k for the LLM.

### Interview-style code

In [ ]:
def retrieve_pipeline(query, dense_search, rerank_fn, fetch_k=20, top_k=5):
    """Classic RAG retrieval pattern."""
    candidates = dense_search(query, k=fetch_k)   # wide net
    ranked = rerank_fn(query, candidates)       # expensive, accurate
    return ranked[:top_k]                       # context for LLM

# Our config (policy queries use a larger first stage)
FETCH_K = 20
FETCH_K_POLICY = 35
TOP_K = 5
print({"normal": FETCH_K, "policy": FETCH_K_POLICY, "final": TOP_K})

### Applied / impact
- `RERANK_CANDIDATES=20`, `RERANK_CANDIDATES_POLICY=35` in `app/core/config.py`
- Without this, hybrid/rerank never see gold clauses sitting just outside top-5 of pure embedding rank

## 6. Cross-encoder reranking

### Definition
Bi-encoder (embedding): encode query & doc **separately** (fast ANN).  
Cross-encoder: score **(query, doc)** pairs jointly (slower, more accurate). Typical pattern: retrieve 20 → rerank → keep 5.

### Interview-style code

In [ ]:
def fake_cross_encoder_score(query: str, doc: str) -> float:
    """Stand-in for CrossEncoder.predict([(q, d)])."""
    q, d = set(query.lower().split()), set(doc.lower().split())
    return len(q & d) / (len(q) + 1e-6)

def rerank(query: str, docs: list[str], top_n=5) -> list[str]:
    scored = sorted(
        ((fake_cross_encoder_score(query, d), d) for d in docs),
        reverse=True,
    )
    return [d for _, d in scored[:top_n]]

cands = [
    "Q1. How can I get in touch with other members? Ans. Send Interest.",
    "Q1. How can I register? Ans. Fill the sign-up form and Submit.",
    "1.1 No Refunds for subscription plans...",
]
print(rerank("how can I join vedic marriage", cands, top_n=2))

### Applied / impact
- `app/services/reranker_service.py` — `cross-encoder/ms-marco-MiniLM-L-6-v2`
- Extras: length penalty, diversity / per-source cap, Q&A bonus (disabled on **policy** queries), legal bonus on policy queries
- Registration intent: prefer registration FAQ over member-contact for join/register queries
- Live fix: join → registration answer, not Interest/contact spam

## 7. Query intent (generic, not doc hardcoding)

### Definition
Light classification of query *class* (policy vs registration) to change **retrieval knobs** — never map to a hardcoded chunk ID.

### Interview-style code

In [ ]:
import re

POLICY = re.compile(r"\b(refund|terms|policy|cancel|jurisdiction|force majeure)\b", re.I)
REGISTER = re.compile(r"\b(register|sign\s*up|join|create account)\b", re.I)

def is_policy(q: str) -> bool:
    return bool(POLICY.search(q))

def is_registration(q: str) -> bool:
    return bool(REGISTER.search(q))

def fetch_k_for(q: str) -> int:
    return 35 if is_policy(q) else 20

for q in ["how can I join?", "are there refund exceptions?"]:
    print(q, "→ fetch_k", fetch_k_for(q), "policy", is_policy(q))

### Applied / impact
- `app/services/query_intent.py`
- Policy queries: larger fetch_k, no FAQ Q&A bonus, soft legal boost
- **Not** “if EU then load clause 3” — no hardcoded solutions

## 8. Hybrid retrieval: BM25 + embeddings (weighted RRF)

### Definition
- **Dense:** semantic (“EU rules” ≈ “jurisdiction exceptions”)
- **BM25:** keyword / rare legal tokens (`1.1`, `non-refundable`, `force majeure`)
- **RRF:** fuse ranks without calibrating raw scores  
  `score(d) = Σ_list w_list / (k + rank_list(d))`

### Interview-style code

In [ ]:
from collections import defaultdict

def reciprocal_rank_fusion(ranked_lists, weights, k=60):
    """Weighted RRF — standard interview hybrid answer."""
    scores = defaultdict(float)
    first = {}
    for w, ranked in zip(weights, ranked_lists):
        for r, doc_id in enumerate(ranked, start=1):
            scores[doc_id] += w * (1.0 / (k + r))
            first.setdefault(doc_id, doc_id)
    return [d for d, _ in sorted(scores.items(), key=lambda x: -x[1])]

# Suppose dense vs BM25 order for a Terms query
dense = ["privacy", "force_majeure", "refund_1_1", "dispute", "intro"]
bm25  = ["force_majeure", "refund_1_1", "privacy", "cancel_2_1", "dispute"]

print("50/50", reciprocal_rank_fusion([dense, bm25], [0.5, 0.5])[:3])
print("30/70", reciprocal_rank_fusion([dense, bm25], [0.3, 0.7])[:3])

### Applied in our project
- `app/services/hybrid_retriever.py` — `BM25Index`, `hybrid_search`, weighted RRF
- Wired in `RerankRetriever` (`rag_service.py`)
- Eval: `scripts/eval_hybrid_retrieval.py --sweep`

### Impact (qwen3 store, strengthened qrels, top_k=5)

| Mode | Terms MRR | Refund MRR | FAQ MRR | ALL MRR |
|------|----------:|-----------:|--------:|--------:|
| Dense only | 0.585 | **0.875** | 0.938 | 0.751 |
| Hybrid 50/50 | **0.641** | 0.796 | 0.942 | 0.769 |
| **Hybrid 30/70 (best overall)** | 0.640 | 0.789 | **0.954** | **0.771** |
| Policy-only hybrid | weaker on Terms | — | — | missed many Terms queries |

**Production default:** always-on hybrid, dense **0.3**, BM25 **0.7**.

## 9. Qrels, paraphrases, composites (evaluation ≠ hardcoding)

### Definition
**qrels** = graded relevance for IR metrics.  
- Same `query_id`, multiple rows → **multi-relevant chunks** (e.g. register + free).  
- New `query_id` per paraphrase → test synonym robustness (“join” vs “register”).

Precision@5 with 1 gold chunk is capped at **0.2** when k=5 — low precision with good recall is normal.

### Interview-style code

In [ ]:
def mrr(ranked_ids: list[str], relevant: set[str]) -> float:
    for i, doc_id in enumerate(ranked_ids, start=1):
        if doc_id in relevant:
            return 1.0 / i
    return 0.0

def recall_at_k(ranked_ids: list[str], relevant: set[str], k=5) -> float:
    hit = sum(1 for d in ranked_ids[:k] if d in relevant)
    return hit / max(1, len(relevant))

# Single-relevant chunk
print("single", mrr(["A","B","gold","C","D"], {"gold"}),
      recall_at_k(["A","B","gold","C","D"], {"gold"}))

# Multi-chunk composite (register + free)
gold = {"registration-Q1", "registration-Q3"}
ranked = ["registration-Q1", "noise", "registration-Q3", "x", "y"]
print("composite", mrr(ranked, gold), recall_at_k(ranked, gold))

### Applied / impact
- Generators: FAQ / all-docs / paraphrases / composites (config JSON, not runtime hardcoding)
- `scripts/patch_faq_qrels.py`, `strengthen_legal_qrels.py`, `validate_qrels.py`
- Composites: SUB/REG/ONBOARD/AUTH/REFUND starter set in `composite_queries.json`
- Taught us FAQ ≫ Terms before hybrid; guided 30/70 weight choice

## 10. Why FAQ MRR ≫ Terms MRR (discussion notes)

| FAQ | Terms / Refund |
|-----|----------------|
| Query ≈ question line in chunk | Query paraphrase ↔ legal jargon |
| Short Q&A units | Long lookalike sections |
| Unique `faq_id` match | Need headers + BM25 + legal_id |
| Reranker Q&A bonus helps FAQ | Same bonus can hurt policy if not gated |

After hybrid 30/70: Terms MRR **0.58 → 0.64**, hit rate **74% → 84%**.

## 11. Best configuration yet (cheat sheet)

```text
Embedding store : ollama_qwen3-0.6b  (qwen3-embedding:0.6b)
Chunking        : FAQ by Q&A + faq_id; legal by section/clause + legal_id/headers
Hybrid          : ON, dense_weight=0.3, sparse_weight=0.7, always-on
Candidate pool  : fetch_k=20 (35 if policy-like)
Reranker        : ms-marco-MiniLM-L-6-v2; no Q&A bonus on policy queries
Return to LLM   : top_k=5
```

| Bucket | Best observed | Notes |
|--------|---------------|-------|
| ALL | MRR **0.771** | hybrid 30/70 |
| Terms | MRR **~0.64** | hybrid helps most |
| FAQ | MRR **~0.95** | essentially saturated |
| Refund | MRR **0.875** dense | hybrid slightly worse rank |

Key files:
- `app/services/document_service.py`
- `app/services/hybrid_retriever.py`
- `app/services/reranker_service.py`
- `app/services/query_intent.py`
- `app/services/rag_service.py`
- `app/core/config.py`
- `scripts/eval_hybrid_retrieval.py`, `benchmark_vector_stores.py`

## 12. Optional: load latest hybrid summary CSV

In [ ]:
from pathlib import Path
import pandas as pd

root = Path("..") if Path("../data/benchmark_results").exists() else Path(".")
br = root / "data" / "benchmark_results"
files = sorted(br.glob("hybrid_eval_summary_*.csv"))
if files:
    df = pd.read_csv(files[-1])
    print("Loaded", files[-1].name)
    display(df[df.bucket.isin(["Terms", "Refund", "FAQ", "ALL"])])
else:
    print("No hybrid_eval_summary_*.csv found — run:")
    print("  python scripts/eval_hybrid_retrieval.py --sweep")

## 13. Interview Q&A (quick)

**Q: Precision@5 is ~0.13 but Recall is ~0.65 — is retrieval broken?**  
A: Not if most queries have **1** relevant chunk and k=5. Perfect hit ⇒ Precision=0.2. Mean Precision ≈ Recall × (1/k).

**Q: Why not hardcode “join → registration FAQ”?**  
A: Doesn’t generalize; breaks when docs change. Fix chunk IDs, hybrid, rerank instead. Use composites only in **eval**.

**Q: Bi-encoder vs cross-encoder?**  
A: Bi-encoder = scalable ANN; cross-encoder = accurate pair scoring on a shortlist.

**Q: Why weighted RRF not raw score blend?**  
A: Dense cosine and BM25 scores live on different scales; ranks fuse cleanly.

**Q: Retrieved chunks look duplicated — cleanup bug?**  
A: Not always. Same wording can appear in multiple valid contexts. Dedupe / MMR / per-source caps at retrieve time (see §14).

**Q: Next experiments?**  
A: Adaptive hybrid (dense-only on refund); HyDE/query rewrite; parent–child retrieval; domain embedding fine-tune.

## 14. Interview: duplicate / near-duplicate chunks (not a data-cleanup bug)

### Interview answer
When duplicates aren’t bad data but the **same wording appears in multiple valid contexts**, you don’t delete chunks — you **dedupe at retrieval time** and keep **diverse** evidence.

### Why it happens
Same sentence can live in FAQ + Terms + Refund, or two sections that both say “non-refundable.” Retrieval correctly returns near-duplicates because embeddings/BM25 see overlapping text.

### How to handle it (in order of use)

**1. Near-duplicate dedupe after retrieval**  
Compare candidate chunks with:
- exact hash / normalized text
- or similarity (cosine / Jaccard / MinHash)

If two chunks are too similar (e.g. cosine > 0.9), keep the **higher-ranked** one and drop the rest.

**2. Diversity / MMR (Maximal Marginal Relevance)**  
Instead of “top-k by score only,” pick chunks that are:
- relevant to the query, and
- **dissimilar to already selected** chunks

Classic tradeoff: relevance vs novelty.

**3. Per-source / per-section caps**  
Limit e.g. max 2 chunks from the same document or same `faq_id` / section so one repeated phrase doesn’t fill the whole context window.

**4. Context packing with merge**  
If two hits are almost the same, merge into one passage (or keep one and note the other as a second citation), so the LLM isn’t fed the same sentence 3 times.

**5. Don’t over-clean the corpus**  
If the same rule must appear under Registration and Refund for legal reasons, keep both in the index; enforce diversity **at retrieve/rerank**, not by deleting source truth.

### One-liner for interviews
> “We treat it as a **redundancy problem in the candidate set**, not a corpus bug: score for relevance, then **dedupe / MMR / per-source caps** so the prompt gets unique signal.”

### How this maps to our project
We already do a form of this: **reranker + max chunks per source** (and hybrid/rerank) so repeated FAQ/legal wording doesn’t dominate the top-k.
- `app/services/reranker_service.py` → `RERANK_MAX_CHUNKS_PER_SOURCE` (higher for policy queries)
- Structure-aware IDs (`faq_id`, `legal_id`) make “same text, different context” measurable; we do **not** delete repeated legal wording from the corpus

### Interview-style code (from scratch)


In [ ]:
import re
from dataclasses import dataclass


@dataclass
class Hit:
    text: str
    score: float
    source: str


def normalize(text: str) -> str:
    return re.sub(r"\s+", " ", text.lower()).strip()


def jaccard(a: str, b: str) -> float:
    sa, sb = set(normalize(a).split()), set(normalize(b).split())
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


def dedupe_near_duplicates(hits: list[Hit], sim_threshold: float = 0.85) -> list[Hit]:
    """Keep higher-scored hit; drop later near-duplicates."""
    kept: list[Hit] = []
    for h in sorted(hits, key=lambda x: -x.score):
        if any(jaccard(h.text, k.text) >= sim_threshold for k in kept):
            continue
        kept.append(h)
    return kept


def diversify_per_source(hits: list[Hit], max_per_source: int = 2, top_k: int = 5) -> list[Hit]:
    """Cap how many chunks one document/section can contribute."""
    out, counts = [], {}
    for h in sorted(hits, key=lambda x: -x.score):
        n = counts.get(h.source, 0)
        if n >= max_per_source:
            continue
        out.append(h)
        counts[h.source] = n + 1
        if len(out) >= top_k:
            break
    return out


def mmr_select(query_terms: set[str], hits: list[Hit], top_k: int = 5, lambda_: float = 0.7) -> list[Hit]:
    """Toy MMR: relevance to query vs novelty vs already picked."""
    def rel(h: Hit) -> float:
        words = set(normalize(h.text).split())
        return len(query_terms & words) / (len(query_terms) + 1e-6)

    remaining = list(hits)
    selected: list[Hit] = []
    while remaining and len(selected) < top_k:
        best, best_score = None, -1e9
        for h in remaining:
            novelty = 0.0
            if selected:
                novelty = max(jaccard(h.text, s.text) for s in selected)
            score = lambda_ * rel(h) - (1 - lambda_) * novelty
            if score > best_score:
                best, best_score = h, score
        selected.append(best)
        remaining.remove(best)
    return selected


# Demo: same rule repeated across FAQ + Terms
candidates = [
    Hit("All subscription payments are non-refundable once activated.", 0.92, "faq"),
    Hit("All subscription payments are non-refundable once activated. See billing.", 0.88, "terms"),
    Hit("Force majeure events may pause matching services temporarily.", 0.70, "terms"),
    Hit("You can upgrade from Free to Premium anytime in Settings.", 0.65, "faq"),
]
print("deduped:", [(h.source, h.text[:40]) for h in dedupe_near_duplicates(candidates)])
print("per-source:", [(h.source, h.text[:40]) for h in diversify_per_source(candidates)])
q = set("non refundable subscription payment".split())
print("mmr:", [(h.source, h.text[:40]) for h in mmr_select(q, candidates, top_k=3)])

### Applied / impact (recap)
- Stops one repeated FAQ/legal phrase from occupying all of top-k
- Leaves room for complementary chunks (e.g. register + free tier, or force-majeure + refund clause)
- Interview framing: redundancy is handled in the **candidate set**, not by pretending the corpus has only one copy of every sentence


---
*Notebook generated for HR_chat_bot retrieval improvement notes. Soft metrics from `hybrid_eval_summary_20260714_121509.csv` / related runs.*